# Discover a BHoM VIKTOR app

Inspect one configured BHoM app entity, discover its saved and declared inputs, list callable methods, and generate a JSON Schema for the agent workflow.

Method execution is intentionally disabled by default. Set both `RUN_METHOD=true` and `METHOD_NAME=<exact method>` only after reviewing the discovered methods. Button and set-params methods can mutate entity or editor state.

In [1]:
import json
import os
import time
from pathlib import Path
from typing import Any

import requests
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    for candidate in (Path(".env"), Path("../.env")):
        if candidate.is_file():
            dotenv_path = str(candidate.resolve())
            break

if dotenv_path:
    load_dotenv(dotenv_path, override=True)

APPS = {
    "bhom_material_template_mapping": {"workspace_id": 3425, "entity_id": 14969},
}

APP_KEY = os.getenv("BHOM_APP", "bhom_material_template_mapping").strip()
if APP_KEY not in APPS:
    raise ValueError(f"Unknown BHOM_APP={APP_KEY!r}; choose one of {sorted(APPS)}")

WORKSPACE_ID = APPS[APP_KEY]["workspace_id"]
ENTITY_ID = APPS[APP_KEY]["entity_id"]
OUTPUT_DIR = Path("artifacts") / APP_KEY
SCHEMA_FILE = OUTPUT_DIR / "input_schema.json"
METHODS_FILE = OUTPUT_DIR / "available_methods.json"
RESULT_FILE = OUTPUT_DIR / "method_result.json"

print("Loaded .env:", dotenv_path or "not found")
print(f"Selected app: {APP_KEY} (workspace {WORKSPACE_ID}, entity {ENTITY_ID})")

Loaded .env: /Users/alejandroduarte/Documents/bhom-viktor-integration/viktor-bhom-agentic-agent/.env
Selected app: bhom_material_template_mapping (workspace 3425, entity 14969)


## Connect safely

The client normalizes the VIKTOR host once, reads a Personal Access Token from the environment, centralizes timeouts and errors, and never prints the secret. `ENV_VKT` defaults to `demo.viktor.ai`.

In [2]:
TOKEN_ENV_NAMES = ("TOKEN_VK_APP", "VIKTOR_TOKEN", "VIKTOR_API_TOKEN")


def normalize_token(raw_token: str, *, env_name: str) -> str:
    token = raw_token.strip().strip('"').strip("'").strip()
    if token.startswith(f"{env_name}="):
        token = token.split("=", 1)[1].strip().strip('"').strip("'")
    if token.lower().startswith("authorization:"):
        token = token.split(":", 1)[1].strip()
    if token.lower().startswith("bearer "):
        token = token.split(None, 1)[1].strip()
    if not token or any(character.isspace() for character in token):
        raise ValueError(f"{env_name} must contain only the token value")
    return token


def load_token() -> tuple[str, str]:
    for env_name in TOKEN_ENV_NAMES:
        raw_token = os.getenv(env_name)
        if raw_token and raw_token.strip():
            return normalize_token(raw_token, env_name=env_name), env_name
    raise ValueError(f"Set one of {TOKEN_ENV_NAMES} to a VIKTOR Personal Access Token")


def api_base_url() -> str:
    configured = os.getenv("VIKTOR_API_BASE", "").strip().rstrip("/")
    if configured:
        base = configured
    else:
        configured_environment = os.getenv("ENV_VKT") or os.getenv("VIKTOR_ENVIRONMENT")
        environment = (configured_environment or "demo.viktor.ai").strip().rstrip("/")
        if environment.startswith("https://"):
            base = environment
        elif environment.endswith(".viktor.ai"):
            base = f"https://{environment}"
        else:
            base = f"https://{environment}.viktor.ai"
    if base.startswith("http://"):
        base = "https://" + base.removeprefix("http://")
    if not base.startswith("https://"):
        raise ValueError("The VIKTOR API base must resolve to HTTPS")
    return base if base.endswith("/api") else f"{base}/api"


class ViktorRestClient:
    def __init__(self, *, api_base: str, token: str) -> None:
        self.api_base = api_base.rstrip("/")
        self.timeout = (5.0, 30.0)
        self.session = requests.Session()
        self.session.headers.update(
            {"Authorization": f"Bearer {token}", "Accept": "application/json"}
        )

    def url(self, path_or_url: str) -> str:
        if path_or_url.startswith(("http://", "https://")):
            return path_or_url
        return f"{self.api_base}/{path_or_url.lstrip('/')}"

    def request_json(
        self,
        method: str,
        path_or_url: str,
        *,
        params: dict[str, Any] | None = None,
        json_body: dict[str, Any] | None = None,
        action: str,
    ) -> dict[str, Any]:
        response = self.session.request(
            method,
            self.url(path_or_url),
            params=params,
            json=json_body,
            timeout=self.timeout,
        )
        if not response.ok:
            raise RuntimeError(
                f"{action} failed ({response.status_code}): {response.text[:500]}"
            )
        if not response.content:
            return {}
        try:
            payload = response.json()
        except ValueError as exc:
            raise RuntimeError(
                f"{action} did not return JSON: {response.text[:500]}"
            ) from exc
        if not isinstance(payload, dict):
            raise TypeError(
                f"{action} returned {type(payload).__name__}, not an object"
            )
        return payload

    def get_json(
        self, path: str, *, params: dict[str, Any] | None = None, action: str
    ) -> dict[str, Any]:
        return self.request_json("GET", path, params=params, action=action)

    def post_json(
        self, path: str, *, json_body: dict[str, Any] | None = None, action: str
    ) -> dict[str, Any]:
        return self.request_json("POST", path, json_body=json_body, action=action)


api_token, token_source = load_token()
client = ViktorRestClient(api_base=api_base_url(), token=api_token)
print(f"Using token from {token_source}; secret not displayed")
print(f"REST API base: {client.api_base}")

Using token from TOKEN_VK_APP; secret not displayed
REST API base: https://demo.viktor.ai/api


## Read entity inputs and editor metadata

Request saved properties, cleaned parameters, and parameter types. Then create an editor session and request the parametrization so declared defaults and button methods are available.

In [3]:
def extract_saved_params(payload: dict[str, Any]) -> dict[str, Any]:
    for key in ("properties", "clean_params", "params", "last_saved_params"):
        value = payload.get(key)
        if isinstance(value, dict):
            return value
    for key in ("last_saved_revision", "latest_revision", "revision"):
        revision = payload.get(key)
        if isinstance(revision, dict) and isinstance(revision.get("params"), dict):
            return revision["params"]
    raise KeyError(f"No saved parameters found; response keys: {list(payload)}")


entity = client.get_json(
    f"workspaces/{WORKSPACE_ID}/entities/{ENTITY_ID}/",
    params={"properties": "true", "clean_params": "true", "param_types": "true"},
    action="Get entity",
)
saved_params = extract_saved_params(entity)
entity_type_id = entity.get("entity_type")
if isinstance(entity_type_id, dict):
    entity_type_id = entity_type_id.get("id")
if entity_type_id is None:
    raise KeyError(f"Entity response has no entity_type identifier: {list(entity)}")

entity_type = client.get_json(
    f"workspaces/{WORKSPACE_ID}/entity_types/{entity_type_id}/",
    action="Get entity type",
)
editor_session = client.post_json(
    f"workspaces/{WORKSPACE_ID}/entities/{ENTITY_ID}/session/",
    action="Create editor session",
)
session_id = editor_session.get("editor_session")
if not session_id:
    raise KeyError(
        f"Editor session response has no editor_session: {list(editor_session)}"
    )
parametrization = client.post_json(
    f"workspaces/{WORKSPACE_ID}/entities/{ENTITY_ID}/parametrization/",
    json_body={"editor_session": session_id, "params": {}},
    action="Get parametrization",
)
parametrization_nodes = (parametrization.get("content") or {}).get(
    "parametrization", []
)
print(f"Entity: {entity.get('name', '<unnamed>')}")
print(f"Saved top-level inputs: {sorted(saved_params)}")

Entity: Demo
Saved top-level inputs: ['dataset_scope', 'gross_floor_area_m2', 'lookup_results_json', 'mapping_state_json', 'material_inventory', 'takeoff_file', 'takeoff_json', 'template_materials_json']


## Merge defaults and discover methods

Declared parametrization defaults are merged with saved entity inputs, with saved values taking precedence. View methods and parametrization actions are deduplicated by method name and saved for review.

In [4]:
def set_nested(target: dict[str, Any], dotted_path: str, value: Any) -> None:
    keys = [part for part in dotted_path.split(".") if part]
    if not keys:
        return
    current = target
    for key in keys[:-1]:
        child = current.setdefault(key, {})
        if not isinstance(child, dict):
            child = {}
            current[key] = child
        current = child
    current[keys[-1]] = value


def collect_declared_defaults(nodes: Any) -> dict[str, Any]:
    defaults: dict[str, Any] = {}

    def walk(value: Any) -> None:
        if isinstance(value, list):
            for item in value:
                walk(item)
        elif isinstance(value, dict):
            path = value.get("name") or value.get("parametrization_path")
            if path and "default" in value:
                set_nested(defaults, path, value["default"])
            for child_key in ("content", "children", "items"):
                walk(value.get(child_key))

    walk(nodes)
    return defaults


def deep_merge(defaults: Any, overrides: Any) -> Any:
    if isinstance(defaults, dict) and isinstance(overrides, dict):
        merged = dict(defaults)
        for key, value in overrides.items():
            merged[key] = deep_merge(merged.get(key), value)
        return merged
    return overrides if overrides is not None else defaults


def collect_view_methods(payload: dict[str, Any]) -> list[dict[str, Any]]:
    methods = []
    for view in payload.get("views") or []:
        if not isinstance(view, dict):
            continue
        name = view.get("controller_method") or view.get("method_name")
        if name:
            methods.append(
                {
                    "method_name": name,
                    "source": "entity_type.views",
                    "label": view.get("label"),
                    "view_type": view.get("view_type"),
                }
            )
    return methods


def collect_parametrization_methods(nodes: Any) -> list[dict[str, Any]]:
    methods: list[dict[str, Any]] = []

    def walk(value: Any) -> None:
        if isinstance(value, list):
            for item in value:
                walk(item)
        elif isinstance(value, dict):
            name = value.get("method")
            if name:
                methods.append(
                    {
                        "method_name": name,
                        "source": "parametrization",
                        "label": value.get("ui_name") or value.get("title"),
                        "node_type": value.get("type"),
                        "path": value.get("name") or value.get("parametrization_path"),
                    }
                )
            for child_key in ("content", "children", "items"):
                walk(value.get(child_key))

    walk(nodes)
    return methods


def deduplicate_methods(methods: list[dict[str, Any]]) -> list[dict[str, Any]]:
    by_name: dict[str, dict[str, Any]] = {}
    for method in methods:
        by_name.setdefault(str(method["method_name"]), method)
    return list(by_name.values())


declared_defaults = collect_declared_defaults(parametrization_nodes)
effective_params = deep_merge(declared_defaults, saved_params)
available_methods = deduplicate_methods(
    collect_view_methods(entity_type)
    + collect_parametrization_methods(parametrization_nodes)
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METHODS_FILE.write_text(json.dumps(available_methods, indent=2), encoding="utf-8")
print(f"Declared default keys: {sorted(declared_defaults)}")
print(f"Effective input keys: {sorted(effective_params)}")
print(json.dumps(available_methods, indent=2, default=str))
print(f"Saved method inventory to {METHODS_FILE.resolve()}")

Declared default keys: ['dataset_scope', 'gross_floor_area_m2', 'lookup_results_json', 'mapping_state_json', 'material_inventory', 'takeoff_json', 'template_materials_json']
Effective input keys: ['dataset_scope', 'gross_floor_area_m2', 'lookup_results_json', 'mapping_state_json', 'material_inventory', 'takeoff_file', 'takeoff_json', 'template_materials_json']
[
  {
    "method_name": "mapping_view",
    "source": "entity_type.views",
    "label": "Mapping",
    "view_type": "web"
  },
  {
    "method_name": "template_view",
    "source": "entity_type.views",
    "label": "Material template",
    "view_type": "data"
  },
  {
    "method_name": "workflow_handoff_view",
    "source": "entity_type.views",
    "label": "Workflow handoff",
    "view_type": "data"
  },
  {
    "method_name": "search_bhom_database",
    "source": "parametrization",
    "label": "Search installed BHoM datasets",
    "node_type": "set-params-button",
    "path": "search_database"
  },
  {
    "method_name": "do

## Infer and save the input schema

The schema reflects effective parameters, including declared defaults overridden by values already saved on the selected entity.

In [5]:
def infer_value_schema(value: Any, *, include_default: bool = True) -> dict[str, Any]:
    if isinstance(value, bool):
        schema: dict[str, Any] = {"type": "boolean"}
    elif isinstance(value, int):
        schema = {"type": "integer"}
    elif isinstance(value, float):
        schema = {"type": "number"}
    elif isinstance(value, str):
        schema = {"type": "string"}
    elif isinstance(value, dict):
        schema = infer_object_schema(value, include_defaults=include_default)
    elif isinstance(value, list):
        if not value:
            item_schema = {}
        elif all(isinstance(item, dict) for item in value):
            shape: dict[str, Any] = {}
            for item in value:
                shape = deep_merge(shape, item)
            item_schema = infer_object_schema(shape, include_defaults=False)
        else:
            item_schema = infer_value_schema(value[0], include_default=False)
        schema = {"type": "array", "items": item_schema}
    elif value is None:
        schema = {"type": ["string", "null"]}
    else:
        schema = {}
    if include_default:
        schema["default"] = value
    return schema


def infer_object_schema(
    values: dict[str, Any], *, include_defaults: bool = True
) -> dict[str, Any]:
    return {
        "type": "object",
        "properties": {
            key: infer_value_schema(value, include_default=include_defaults)
            for key, value in values.items()
        },
        "additionalProperties": False,
    }


schema = infer_object_schema(effective_params)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCHEMA_FILE.write_text(json.dumps(schema, indent=2, default=str), encoding="utf-8")
print(json.dumps(schema, indent=2, default=str))
print(f"Saved input schema to {SCHEMA_FILE.resolve()}")

{
  "type": "object",
  "properties": {
    "dataset_scope": {
      "type": "string",
      "default": "All installed LCA datasets"
    },
    "gross_floor_area_m2": {
      "type": "integer",
      "default": 0
    },
    "material_inventory": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "material_name": {
            "type": "string"
          },
          "search_query": {
            "type": "string"
          },
          "volume_m3": {
            "type": "number"
          },
          "density_kg_m3": {
            "type": "number"
          }
        },
        "additionalProperties": false
      },
      "default": [
        {
          "material_name": "Concrete C30/37",
          "search_query": "ready mix concrete C30/37",
          "volume_m3": 32.0,
          "density_kg_m3": 2400.0
        },
        {
          "material_name": "Structural Steel",
          "search_query": "structural steel",
          "volume_m

## Optional explicit method execution

Nothing runs unless `RUN_METHOD=true`. You must also set `METHOD_NAME` to an exact name from the method inventory. Review its source and type first: parametrization buttons and set-params methods may change persisted or editor-session state.

In [6]:
RESULT_KEY_PRIORITY = [
    "data",
    "table",
    "download",
    "geometry",
    "plotly",
    "geojson",
    "web",
    "pdf",
    "image",
    "ifc",
    "optimization",
    "set_params",
]


def execute_method(method_name: str) -> dict[str, Any]:
    job = client.post_json(
        f"workspaces/{WORKSPACE_ID}/entities/{ENTITY_ID}/jobs/",
        json_body={
            "method_name": method_name,
            "params": effective_params,
            "poll_result": False,
        },
        action="Create VIKTOR job",
    )
    job_url = job.get("url")
    deadline = time.monotonic() + 300
    delay = 0.8
    while job_url and time.monotonic() < deadline:
        job = client.get_json(job_url, action="Poll VIKTOR job")
        status = job.get("status")
        if status == "success":
            break
        if status in {"failed", "cancelled", "error", "error_user", "error_timeout"}:
            raise RuntimeError(
                f"VIKTOR job failed with status={status}: {job.get('error')}"
            )
        time.sleep(delay)
        delay = min(delay * 1.5, 5.0)
    else:
        if job_url:
            raise TimeoutError("VIKTOR job did not finish within 300 seconds")
    if job.get("status") != "success":
        raise RuntimeError(f"Unexpected VIKTOR job response: {job}")
    result = job.get("result") or job.get("content")
    if not isinstance(result, dict):
        raise TypeError(f"Method result is not a JSON object: {job}")
    return result


run_method = os.getenv("RUN_METHOD", "false").strip().lower() == "true"
method_name = os.getenv("METHOD_NAME", "").strip()
if not run_method:
    print(
        "Method execution skipped; set RUN_METHOD=true and METHOD_NAME explicitly to opt in"
    )
elif not method_name:
    raise ValueError(
        "METHOD_NAME is required when RUN_METHOD=true; choose from available_methods"
    )
else:
    known_names = {str(method["method_name"]) for method in available_methods}
    if method_name not in known_names:
        raise ValueError(
            f"METHOD_NAME={method_name!r} was not discovered; choose from {sorted(known_names)}"
        )
    method_result = execute_method(method_name)
    result_key = next(
        (key for key in RESULT_KEY_PRIORITY if key in method_result), "result"
    )
    selected_result = method_result.get(result_key, method_result)
    stored_result = {
        "method_name": method_name,
        "result_key": result_key,
        "result": selected_result,
    }
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    RESULT_FILE.write_text(
        json.dumps(stored_result, indent=2, default=str), encoding="utf-8"
    )
    print(
        f"Executed {method_name}; saved {result_key!r} payload to {RESULT_FILE.resolve()}"
    )

Method execution skipped; set RUN_METHOD=true and METHOD_NAME explicitly to opt in
